# Qwen3-Reranker: score every reasoning trace against its claim and evidence

**What this notebook does.** For every claim it asks Qwen3-Reranker one yes/no question per trace:
*"Given this claim and this evidence, does this reasoning trace reach the correct verdict using facts supported by the evidence?"*
The answer is a score per trace (`logit` = log-odds of "yes", and `p_yes`). It does **not** use the gold label, so it is a
zero-shot, trace-level relevance signal that reads the evidence, which the current p/n ranker never does.

**Speed trick (prefix caching).** The claim + evidence is the same for all 20 traces of a claim. It is run through the
model once, its key/value cache is copied for the traces, and only the trace tokens are computed again. A self-test cell
checks that this gives the same scores as the plain way; if not, the notebook falls back to the plain way.

**Two GPUs.** One copy of the model per T4; the two GPUs take shards of claims from a shared queue.

**Resumable.** Results are written in shards of 32 claims. If the 12-hour session ends, save the version, add its output
as an input of this notebook and run again: finished shards are found and skipped.

**Outputs** (in `/kaggle/working/<TAG>/`): `{language}_{split}_<TAG>.npz` with `logit` and `p_yes` of shape
(claims, 20), NaN where a claim has fewer traces, aligned with `Reasoning_traces` / `Verdict_list` order; plus `evaluation.csv`.

**Cells:** 1 Config · 2 Setup + data · 3 Model and scoring · 4 Self-test and time estimate · 5 Run · 6 Merge · 7 Evaluation

In [1]:
# ───────────────────────────── 1. Config ─────────────────────────────
MODEL_ID = "Qwen/Qwen3-Reranker-0.6B"     # also: "Qwen/Qwen3-Reranker-4B" (fp16, fits one T4), "Qwen/Qwen3-Reranker-8B" (set LOAD_4BIT = True)
LOAD_4BIT = False
TAG = "qwen3rr06b_v1"                     # change when you change the model or the instruction
LANGS = ["english", "spanish", "arabic"]
SPLITS = ["test", "val", "train"]         # test first: it matters most; train is needed later for learning-to-rank
LIMIT_CLAIMS = None                       # e.g. 40 for a quick trial (random claims per split; TAG gets "_trial")
MAX_QUERY_TOKENS = 6144                   # claim + evidence (test evidence: median ~2.2k tokens, max ~5.6k)
MAX_DOC_TOKENS = 1536                     # one reasoning trace
TOKENS_PER_PASS = None                    # None = automatic by model size (traces x tokens computed in one forward pass)
SHARD_SIZE = 32                           # claims per saved shard
TIME_BUDGET_HOURS = 11.0                  # stop taking new shards after this (Kaggle sessions end at 12 h)
DATA_ROOT = "/kaggle/input"               # searched recursively for the CheckThat! json files
RESUME_SEARCH_ROOT = "/kaggle/input"      # finished shards from an earlier run are looked for here
OUTPUT_DIR = "/kaggle/working"
INSTRUCTION = ("Given a numerical claim and the evidence retrieved for it, judge whether the reasoning trace reaches the correct "
               "verdict about the claim (True, False or Conflicting), using only facts that are supported by the evidence.")

In [2]:
# ───────────────────────────── 2. Setup + data ─────────────────────────────
import os, re, gc, sys, json, glob, copy, time, math, queue, shutil, random, threading
import numpy as np, pandas as pd, torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
print(f"PyTorch {torch.__version__} | transformers {transformers.__version__} | GPUs {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  {torch.cuda.get_device_properties(i).total_memory / 2**30:.1f} GiB")
N_GPUS = max(torch.cuda.device_count(), 1)
if LIMIT_CLAIMS: TAG += "_trial"
WORK = os.path.join(OUTPUT_DIR, TAG); SHARDS = os.path.join(WORK, "shards"); os.makedirs(SHARDS, exist_ok=True)
T_START = time.time()

FILES = {"english": ("English", ["train.json", "validation.json", "test.json"]),
         "spanish": ("Spanish", ["spanish_train.json", "spanish_val.json", "clef_spanish_test_final_with_gold_labels.json"]),
         "arabic": ("Arabic", ["clef2026_gpt4_o_mini_train_arabic.json", "clef2026_gpt4_o_mini_val_arabic.json", "clef_arabic_test_gold_labels-final.json"])}

def find_file(folder, name):
    hits = [p for p in glob.glob(os.path.join(DATA_ROOT, "**", name), recursive=True) if f"{os.sep}{folder}{os.sep}" in os.path.normpath(p)]
    if not hits: raise FileNotFoundError(f"{folder}/{name} not found under {DATA_ROOT}")
    return sorted(hits, key=len)[0]

DATA = {}
for lang in LANGS:
    folder, names = FILES[lang]
    for split, name in zip(["train", "val", "test"], names):
        if split not in SPLITS: continue
        path = find_file(folder, name); items = json.load(open(path, encoding="utf-8"))
        rows = list(range(len(items)))
        if LIMIT_CLAIMS: rows = sorted(random.Random(0).sample(rows, min(LIMIT_CLAIMS, len(rows))))
        DATA[lang, split] = {"items": items, "rows": rows, "path": path}
        nt = [len(it["Reasoning_traces"]) for it in items]
        print(f"{lang:8s} {split:5s}: {len(items):5d} claims ({len(rows)} scored) | traces/claim mean {np.mean(nt):.1f} max {max(nt)} | {path}")

def query_text(it):
    ev = it.get("evidences") or []
    ev = "\n".join(f"[{i + 1}] {str(e).strip()}" for i, e in enumerate(ev)) if ev else "(no evidence)"
    return f"Claim: {it['claim'].strip()}\n\nEvidence:\n{ev}"

PyTorch 2.11.0+cu130 | transformers 5.20.2 | GPUs 2
  GPU 0: Tesla T4  14.6 GiB
  GPU 1: Tesla T4  14.6 GiB
english  train:  6400 claims (6400 scored) | traces/claim mean 19.8 max 20 | /kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/English/train.json
english  val  :  1600 claims (1600 scored) | traces/claim mean 15.0 max 15 | /kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/English/validation.json
english  test :  2558 claims (2558 scored) | traces/claim mean 20.0 max 20 | /kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/English/test.json
spanish  train:  2246 claims (2246 scored) | traces/claim mean 20.0 max 20 | /kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/Spanish/spanish_train.json
spanish  val  :   562 claims (562 scored) | traces/claim mean 20.0 max 20 | /kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/Spanish/spanish_val.json
spanish  test :  1164 claims (1164 scored) | traces/claim mean 20

In [3]:
# ───────────────────────────── 3. Model and scoring ─────────────────────────────
tok = AutoTokenizer.from_pretrained(MODEL_ID)
PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
ID_NO, ID_YES = tok.convert_tokens_to_ids("no"), tok.convert_tokens_to_ids("yes")
# Prompt format of the Qwen3-Reranker model card: system prefix + <Instruct>/<Query>/<Document> + assistant suffix.
SYS = ("<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. "
       "Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n")
SUF = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
enc = lambda s: tok.encode(s, add_special_tokens=False)
SYS_IDS, SUF_IDS, DOC_TAG = enc(SYS), enc(SUF), enc("\n<Document>: ")
INS_IDS = enc(f"<Instruct>: {INSTRUCTION}\n<Query>: ")
print(f"model {MODEL_ID} | token ids no={ID_NO} yes={ID_YES} | pad {PAD}")

def build_ids(it):
    """Shared prefix (system + instruction + claim/evidence) and one suffix (trace + assistant turn) per trace; None if empty."""
    q = enc(query_text(it))[:MAX_QUERY_TOKENS]
    prefix = SYS_IDS + INS_IDS + q + DOC_TAG
    sufs = [(enc(t.strip())[:MAX_DOC_TOKENS] + SUF_IDS) if isinstance(t, str) and t.strip() else None for t in it["Reasoning_traces"][:20]]
    return prefix, sufs

if TOKENS_PER_PASS is None:
    TOKENS_PER_PASS = 8192 if ("8B" in MODEL_ID) else 12288 if ("4B" in MODEL_ID) else 32768

def load_model(dev):
    kw = dict(torch_dtype=torch.float16, attn_implementation="sdpa", device_map={"": dev})
    if LOAD_4BIT:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw).eval()
    w = m.lm_head.weight.detach()[[ID_NO, ID_YES]].float()          # only the "no" / "yes" rows of the output layer are needed
    return m, w

def _head(m, w, h):                     # h: (n, hidden) final hidden state at the last real token -> log-odds of "yes"
    l = h.float() @ w.T
    return (l[:, 1] - l[:, 0])

def right_pad(seqs, dev):
    L = max(len(s) for s in seqs); ids = torch.full((len(seqs), L), PAD, dtype=torch.long)
    for i, s in enumerate(seqs): ids[i, :len(s)] = torch.tensor(s)
    return ids.to(dev), torch.tensor([len(s) - 1 for s in seqs], device=dev)

@torch.no_grad()
def score_plain(m, w, prefix, sufs, dev):
    """Reference path: prefix + trace as one sequence. Right padding needs no mask (causal attention never looks ahead)."""
    out = [math.nan] * len(sufs); todo = [j for j, s in enumerate(sufs) if s is not None]
    n = max(1, TOKENS_PER_PASS // (len(prefix) + max([len(sufs[j]) for j in todo] or [1])) // 4)
    for a in range(0, len(todo), n):
        js = todo[a:a + n]; ids, last = right_pad([prefix + sufs[j] for j in js], dev)
        h = m.model(input_ids=ids, use_cache=False).last_hidden_state
        for j, v in zip(js, _head(m, w, h[torch.arange(len(js), device=dev), last]).tolist()): out[j] = v
    return out

@torch.no_grad()
def score_cached(m, w, prefix, sufs, dev):
    """Fast path: run the shared prefix once, repeat its key/value cache for a chunk of traces, run only the trace tokens."""
    out = [math.nan] * len(sufs); todo = [j for j, s in enumerate(sufs) if s is not None]
    if not todo: return out
    P = len(prefix)
    cache = m.model(input_ids=torch.tensor([prefix], device=dev), use_cache=True).past_key_values
    n = max(1, min(len(todo), TOKENS_PER_PASS // (P + max(len(sufs[j]) for j in todo))))
    a = 0
    while a < len(todo):
        js = todo[a:a + n]
        try:
            c = copy.deepcopy(cache); c.batch_repeat_interleave(len(js))
            ids, last = right_pad([sufs[j] for j in js], dev); L = ids.shape[1]
            pos = torch.arange(P, P + L, device=dev)
            h = m.model(input_ids=ids, past_key_values=c, use_cache=True, position_ids=pos.unsqueeze(0).expand(len(js), -1),
                        cache_position=pos).last_hidden_state
            for j, v in zip(js, _head(m, w, h[torch.arange(len(js), device=dev), last]).tolist()): out[j] = v
            a += len(js); del c, h
        except torch.cuda.OutOfMemoryError:
            if n == 1: raise
            n = max(1, n // 2); torch.cuda.empty_cache(); print(f"  [{dev}] out of memory, chunk lowered to {n} traces", flush=True)
    return out

USE_PREFIX_CACHE = True
def score_claim(m, w, it, dev):
    prefix, sufs = build_ids(it)
    s = (score_cached if USE_PREFIX_CACHE else score_plain)(m, w, prefix, sufs, dev)
    return s, len(prefix) + sum(len(x) for x in sufs if x is not None)

MODELS = [load_model(f"cuda:{g}" if torch.cuda.is_available() else "cpu") for g in range(N_GPUS)]
print("loaded on", N_GPUS, "device(s) | tokens per pass", TOKENS_PER_PASS, "|", torch.cuda.memory_allocated(0) / 2**30 if torch.cuda.is_available() else 0, "GiB on GPU 0")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (5 outputs left out)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

loaded on 2 device(s) | tokens per pass 32768 | 1.1097421646118164 GiB on GPU 0


In [4]:
# ───────────────────────────── 4. Self-test and time estimate ─────────────────────────────
# (a) The cached path must give the same scores as the plain path (fp16 noise only).
dev0 = "cuda:0" if torch.cuda.is_available() else "cpu"; m0, w0 = MODELS[0]
key = next(k for k in DATA if k[1] == "test") if any(k[1] == "test" for k in DATA) else next(iter(DATA))
items = DATA[key]["items"]; probe = sorted(DATA[key]["rows"], key=lambda r: -len(query_text(items[r])))[:3]
diffs, t_plain, t_cache, ntok = [], 0.0, 0.0, 0
for r in probe:
    prefix, sufs = build_ids(items[r]); sufs = sufs[:6]
    torch.cuda.synchronize() if torch.cuda.is_available() else None; t0 = time.time(); a = score_plain(m0, w0, prefix, sufs, dev0)
    torch.cuda.synchronize() if torch.cuda.is_available() else None; t1 = time.time(); b = score_cached(m0, w0, prefix, sufs, dev0)
    torch.cuda.synchronize() if torch.cuda.is_available() else None; t2 = time.time()
    t_plain += t1 - t0; t_cache += t2 - t1; ntok += len(prefix) + sum(len(s) for s in sufs if s is not None)
    diffs += [abs(x - y) for x, y in zip(a, b) if not (math.isnan(x) or math.isnan(y))]
    print(f"  {key[0]} {key[1]} row {r}: prefix {len(prefix)} tokens | plain {np.round(a, 2).tolist()}\n{'':38s}cached {np.round(b, 2).tolist()}")
print(f"max |plain - cached| = {max(diffs):.4f} | time plain {t_plain:.1f}s vs cached {t_cache:.1f}s")
if max(diffs) > 0.25:
    USE_PREFIX_CACHE = False; print("!! cached scores differ from the plain path: falling back to the plain path (slower).")
else:
    print("prefix caching OK")

# (b) Time estimate from the token counts (prefix counted once per claim when caching).
tps = ntok / max(t_cache if USE_PREFIX_CACHE else t_plain, 1e-6)
total = 0
for (lang, split), d in DATA.items():
    n = 0
    for r in d["rows"]:
        prefix, sufs = build_ids(d["items"][r]); sl = sum(len(s) for s in sufs if s is not None)
        n += (len(prefix) + sl) if USE_PREFIX_CACHE else sum(len(prefix) + len(s) for s in sufs if s is not None)
    total += n; print(f"  {lang:8s} {split:5s}: {n / 1e6:7.1f} M tokens computed")
print(f"total {total / 1e6:.0f} M tokens | measured ~{tps:,.0f} tokens/s per GPU (probe, rough) | "
      f"estimate {total / tps / N_GPUS / 3600:.1f} h on {N_GPUS} GPU(s); budget per session {TIME_BUDGET_HOURS} h")

  english test row 1690: prefix 5610 tokens | plain [10.17, 9.11, 7.78, 10.67, 9.7, 10.42]
                                      cached [10.15, 9.08, 7.82, 10.65, 9.69, 10.44]
  english test row 53: prefix 5359 tokens | plain [5.05, 5.31, 4.43, 5.32, 4.03, 6.41]
                                      cached [5.05, 5.31, 4.46, 5.34, 4.02, 6.43]
  english test row 1228: prefix 4952 tokens | plain [8.29, 8.55, 8.49, 8.45, 8.5, 8.81]
                                      cached [8.29, 8.55, 8.49, 8.44, 8.49, 8.81]
max |plain - cached| = 0.0371 | time plain 92.1s vs cached 16.2s
prefix caching OK
  english  train:    34.9 M tokens computed
  english  val  :     6.8 M tokens computed
  english  test :    13.7 M tokens computed
  spanish  train:     9.4 M tokens computed
  spanish  val  :     2.4 M tokens computed
  spanish  test :     7.3 M tokens computed
  arabic   train:    14.4 M tokens computed
  arabic   val  :     3.6 M tokens computed
  arabic   test :     1.5 M tokens computed
total 

In [5]:
# ───────────────────────────── 5. Run (both GPUs, resumable) ─────────────────────────────
def shard_name(lang, split, k): return f"{lang}_{split}_{k:04d}.npz"
def find_done(name):
    p = os.path.join(SHARDS, name)
    if os.path.exists(p): return True
    hits = glob.glob(os.path.join(RESUME_SEARCH_ROOT, "**", TAG, "shards", name), recursive=True)
    if hits: shutil.copy(hits[0], p); return True
    return False

JOBS = queue.Queue(); n_done = n_todo = 0
for split in SPLITS:                       # order: test, val, train; inside a split: languages in LANGS order
    for lang in LANGS:
        if (lang, split) not in DATA: continue
        rows = DATA[lang, split]["rows"]
        for k in range(math.ceil(len(rows) / SHARD_SIZE)):
            if find_done(shard_name(lang, split, k)): n_done += 1
            else: JOBS.put((lang, split, k)); n_todo += 1
print(f"shards already done {n_done} | to do {n_todo}")
LOCK, ERRORS, PROG = threading.Lock(), [], {"shards": 0, "tokens": 0}

def worker(g):
    dev = f"cuda:{g}" if torch.cuda.is_available() else "cpu"; m, w = MODELS[g]
    try:
        while time.time() - T_START < TIME_BUDGET_HOURS * 3600:
            try: lang, split, k = JOBS.get_nowait()
            except queue.Empty: return
            d = DATA[lang, split]; rows = d["rows"][k * SHARD_SIZE:(k + 1) * SHARD_SIZE]; t0 = time.time()
            logit = np.full((len(rows), 20), np.nan, np.float32); ntok = 0
            for i, r in enumerate(rows):
                s, nt = score_claim(m, w, d["items"][r], dev); logit[i, :len(s)] = s; ntok += nt
            np.savez(os.path.join(SHARDS, shard_name(lang, split, k)), rows=np.array(rows), logit=logit)
            with LOCK:
                PROG["shards"] += 1; PROG["tokens"] += ntok; el = time.time() - T_START
                print(f"[GPU {g}] {lang:8s} {split:5s} shard {k:4d} | {len(rows)} claims | {ntok / 1e3:6.0f}k tok | {time.time() - t0:5.0f}s | "
                      f"done {PROG['shards']}/{n_todo} | {PROG['tokens'] / el:,.0f} tok/s total | {el / 3600:.2f} h", flush=True)
    except Exception as e:
        import traceback; ERRORS.append((g, traceback.format_exc()))

threads = [threading.Thread(target=worker, args=(g,)) for g in range(N_GPUS)]
for t in threads: t.start()
for t in threads: t.join()
if ERRORS:
    for g, tb in ERRORS: print(f"===== error on GPU {g} =====\n{tb}")
    raise RuntimeError("a worker failed, see the traceback above")
print("left in queue:", JOBS.qsize(), "(0 = everything scored)" if JOBS.qsize() == 0 else "-> time budget reached: save this version and run again with it as input")

shards already done 0 | to do 575
[GPU 1] english  test  shard    1 | 32 claims |    184k tok |    76s | done 1/575 | 357 tok/s total | 0.14 h
[GPU 0] english  test  shard    0 | 32 claims |    191k tok |    90s | done 2/575 | 710 tok/s total | 0.15 h
[GPU 1] english  test  shard    2 | 32 claims |    180k tok |    73s | done 3/575 | 945 tok/s total | 0.16 h
[GPU 0] english  test  shard    3 | 32 claims |    181k tok |    82s | done 4/575 | 1,205 tok/s total | 0.17 h
[GPU 1] english  test  shard    4 | 32 claims |    212k tok |    66s | done 5/575 | 1,450 tok/s total | 0.18 h
[GPU 0] english  test  shard    5 | 32 claims |    199k tok |    56s | done 6/575 | 1,721 tok/s total | 0.19 h
[GPU 1] english  test  shard    6 | 32 claims |    171k tok |    65s | done 7/575 | 1,834 tok/s total | 0.20 h
[GPU 0] english  test  shard    7 | 32 claims |    177k tok |    76s | done 8/575 | 2,015 tok/s total | 0.21 h
[GPU 1] english  test  shard    8 | 32 claims |    167k tok |    70s | done 9/575 | 

In [6]:
# ───────────────────────────── 6. Merge ─────────────────────────────
status = []
for (lang, split), d in DATA.items():
    rows = d["rows"]; ks = range(math.ceil(len(rows) / SHARD_SIZE)); names = [shard_name(lang, split, k) for k in ks]
    have = [n for n in names if find_done(n)]
    if len(have) < len(names):
        status.append((lang, split, f"NOT FINISHED ({len(have)}/{len(names)} shards)")); continue
    logit = np.full((len(d["items"]), 20), np.nan, np.float32)
    for n in names:
        z = np.load(os.path.join(SHARDS, n)); logit[z["rows"]] = z["logit"]
    qid = np.array([str(it.get("query_id", it.get("query_index", r))) for r, it in enumerate(d["items"])])
    out = os.path.join(WORK, f"{lang}_{split}_{TAG}.npz")
    np.savez(out, logit=logit, p_yes=1 / (1 + np.exp(-logit)), scored_rows=np.array(rows), query_id=qid,
             meta=json.dumps({"model": MODEL_ID, "instruction": INSTRUCTION, "max_query_tokens": MAX_QUERY_TOKENS, "max_doc_tokens": MAX_DOC_TOKENS,
                              "prefix_cache": USE_PREFIX_CACHE, "source": d["path"]}))
    status.append((lang, split, f"done -> {out}"))
for s in status: print(f"{s[0]:8s} {s[1]:5s} {s[2]}")

english  train done -> /kaggle/working/qwen3rr06b_v1/english_train_qwen3rr06b_v1.npz
english  val   done -> /kaggle/working/qwen3rr06b_v1/english_val_qwen3rr06b_v1.npz
english  test  done -> /kaggle/working/qwen3rr06b_v1/english_test_qwen3rr06b_v1.npz
spanish  train done -> /kaggle/working/qwen3rr06b_v1/spanish_train_qwen3rr06b_v1.npz
spanish  val   done -> /kaggle/working/qwen3rr06b_v1/spanish_val_qwen3rr06b_v1.npz
spanish  test  done -> /kaggle/working/qwen3rr06b_v1/spanish_test_qwen3rr06b_v1.npz
arabic   train done -> /kaggle/working/qwen3rr06b_v1/arabic_train_qwen3rr06b_v1.npz
arabic   val   done -> /kaggle/working/qwen3rr06b_v1/arabic_val_qwen3rr06b_v1.npz
arabic   test  done -> /kaggle/working/qwen3rr06b_v1/arabic_test_qwen3rr06b_v1.npz


In [7]:
# ───────────────────────────── 7. Evaluation (uses the gold labels, only for checking) ─────────────────────────────
from sklearn.metrics import roc_auc_score
C = ["False", "True", "Conflicting"]
def macro_f1(y, p, labels):
    f = []
    for c in labels:
        tp = sum((a == c) and (b == c) for a, b in zip(y, p)); fp = sum((a != c) and (b == c) for a, b in zip(y, p)); fn = sum((a == c) and (b != c) for a, b in zip(y, p))
        f.append(2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0)
    return float(np.mean(f))
EV = []
for (lang, split), d in DATA.items():
    path = os.path.join(WORK, f"{lang}_{split}_{TAG}.npz")
    if not os.path.exists(path): continue
    L = np.load(path)["logit"]; labels = [c for c in C if any(str(it.get("label", it.get("Label"))) == c for it in d["items"])]
    S, R, auc_in, r5 = [], [], [], {"reranker": [], "dataset order": [], "oracle": []}; y, maj, wv, top1 = [], [], [], []
    for r in d["rows"]:
        it = d["items"][r]; g = str(it.get("label", it.get("Label"))); v = [str(x) for x in it["Verdict_list"][:20]]
        s = L[r, :len(v)]; ok = ~np.isnan(s); rel = np.array([x == g for x in v])
        S += s[ok].tolist(); R += rel[ok].tolist()
        if ok.sum() and 0 < rel[ok].sum() < ok.sum(): auc_in.append(roc_auc_score(rel[ok], s[ok]))
        n_rel = rel.sum()
        for name, order in [("reranker", np.argsort(-np.nan_to_num(s, nan=-1e9), kind="stable")), ("dataset order", np.arange(len(v))),
                            ("oracle", np.argsort(~rel, kind="stable"))]:
            r5[name].append(rel[order[:5]].sum() / n_rel if n_rel else 0.0)
        cnt = {c: sum(x == c for x in v) for c in labels}; wsum = {c: float(np.nansum(np.where(np.array(v) == c, 1 / (1 + np.exp(-s)), 0))) for c in labels}
        y.append(g); maj.append(max(labels, key=lambda c: cnt[c])); wv.append(max(labels, key=lambda c: wsum[c]))
        j = int(np.nanargmax(s)) if ok.any() else 0; top1.append(v[j] if v[j] in labels else labels[0])
    EV.append({"language": lang, "split": split, "claims": len(d["rows"]), "AUROC all traces": roc_auc_score(R, S) if 0 < np.mean(R) < 1 else np.nan,
               "AUROC within claim (mean)": np.mean(auc_in) if auc_in else np.nan,
               **{f"Recall@5 {k}": np.mean(v) for k, v in r5.items()},
               "Macro-F1 majority vote": macro_f1(y, maj, labels), "Macro-F1 score-weighted vote": macro_f1(y, wv, labels), "Macro-F1 top-1 trace": macro_f1(y, top1, labels)})
EV = pd.DataFrame(EV); EV.to_csv(os.path.join(WORK, "evaluation.csv"), index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 20)
print(EV.round(4).to_string(index=False))
print("\nRecall@5 here is over all rows (no claim-text dedup), with 0 for claims that have no correct trace.")
print(f"total time {(time.time() - T_START) / 60:.1f} min")

language split  claims  AUROC all traces  AUROC within claim (mean)  Recall@5 reranker  Recall@5 dataset order  Recall@5 oracle  Macro-F1 majority vote  Macro-F1 score-weighted vote  Macro-F1 top-1 trace
 english train    6400            0.4278                     0.4207             0.1673                  0.1805           0.3323                  0.4478                        0.4428                0.4219
 english   val    1600            0.4486                     0.4020             0.2152                  0.2345           0.3912                  0.4704                        0.4669                0.4237
 english  test    2558            0.6219                     0.5225             0.2212                  0.2175           0.2822                  0.5953                        0.5966                0.5810
 spanish train    2246            0.4683                     0.4233             0.1922                  0.2057           0.3109                  0.3997                        0.3956   

In [8]:
print(0)

0
